# Puzzle #135 production solver (Colab T4)

Points RCKangaroo at the real target (puzzle #135, 134-bit interval, pubkey
exposed in 2019 spending tx). Each Colab session is an independent kangaroo
solve attempt:

1. Mounts Drive (only used to archive a found key + per-session log)
2. Clones project, builds RCKangaroo
3. Loads pubkey/range constants from `kangaroo.puzzle_135`
4. Spawns the solver, streams stdout to a per-session log on Drive
5. If `RESULTS.TXT` appears: runs `verify_solution()` (triple-check —
   interval bracket, on-curve, `d*G == Q`)

**Why no `-tames` checkpointing.** RCKangaroo's `-tames` is for a multi-day
GEN run on serious hardware. The `-max` minimum of 0.001 forces GEN to run
~7 years on T4 before it writes the tames file, so it never completes in a
free-Colab session. We run the full solver each time and accept that state
doesn't persist. Accumulated tames wouldn't change the math at this scale.

**RNG.** RCKangaroo seeds the jump table with 0 (deterministic) but seeds
kangaroo starting positions with `GetTickCount64()`, so each session
explores different walks naturally.

**Reality check.** Expected solve time on a single T4 is ~7,400 years.
This is a lottery ticket. Probability per session ≈ 3×10⁻⁸.

**Runtime:** Runtime → Change runtime type → T4 GPU. Run cells top-to-bottom.

In [ ]:
# Mount Drive (persistent state across sessions).
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Session setup: wipe stale state, fresh clone, ensure Drive layout.
import os, sys, shutil, subprocess

REPO_URL     = "https://github.com/anevolbap/bitcoin-prize.git"
PROJECT_DIR  = "/content/bitcoin-prize"
KANGAROO_DIR = "/content/RCKangaroo"
WORK_DIR     = "/content/work"
DRIVE_DIR    = "/content/drive/MyDrive/kangaroo_135"
SESSIONS_DIR = os.path.join(DRIVE_DIR, "sessions")

for d in (PROJECT_DIR, KANGAROO_DIR):
    if os.path.isdir(d):
        shutil.rmtree(d)

subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR],
    check=True,
)
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(SESSIONS_DIR, exist_ok=True)
sys.path.insert(0, PROJECT_DIR)

# GPU sanity
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
     "--format=csv,noheader"], capture_output=True, text=True,
).stdout)
print(f"project at {PROJECT_DIR}, drive at {DRIVE_DIR}")


In [ ]:
# Build RCKangaroo. Detect nvcc dynamically because the Makefile
# hardcodes a CUDA path that doesn't exist on Colab.
if not os.path.isdir(KANGAROO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/RetiredC/RCKangaroo.git", KANGAROO_DIR],
        check=True,
    )

nvcc_path = shutil.which("nvcc")
if not nvcc_path:
    raise RuntimeError("nvcc not on PATH; check Colab CUDA install")
cuda_root = os.path.dirname(os.path.dirname(nvcc_path))

subprocess.run(["make", "clean"], cwd=KANGAROO_DIR, check=False,
               capture_output=True)
build = subprocess.run(
    ["make", f"CUDA_PATH={cuda_root}", f"NVCC={nvcc_path}"],
    cwd=KANGAROO_DIR, capture_output=True, text=True,
)
if build.returncode != 0:
    print(build.stdout[-1500:])
    print("STDERR:", build.stderr[-1500:])
    raise RuntimeError(f"build failed (rc={build.returncode})")
binary = os.path.join(KANGAROO_DIR, "rckangaroo")
assert os.path.isfile(binary)
print("built:", binary)


In [ ]:
# Load puzzle #135 constants from the project's verified module.
from kangaroo.puzzle_135 import (
    ADDRESS, PUBKEY_HEX, K1, K2, SOURCE_TXID,
)

RANGE_BITS = (K2 - K1).bit_length() - 1   # RCKangaroo's -range = log2(width)
START_HEX = f"{K1:x}"

# DP bits: tune memory vs. lookup overhead.
# RCKangaroo accepts 14..60. For #135 with default walker count, ~16-18 is a
# common starting point. If memory pressure surfaces in a session, raise this.
DP_BITS = 16

print(f"target  : {ADDRESS}")
print(f"pubkey  : {PUBKEY_HEX}")
print(f"range   : -range {RANGE_BITS}  -start {START_HEX}")
print(f"dp_bits : {DP_BITS}")
print(f"source  : tx {SOURCE_TXID}")

In [ ]:
# Wipe any stale local result so this session has clean observability.
LOCAL_RESULT = os.path.join(WORK_DIR, "RESULTS.TXT")
if os.path.exists(LOCAL_RESULT):
    os.remove(LOCAL_RESULT)

In [ ]:
# Spawn RCKangaroo and monitor.
# stdout streams to a per-session log file so we don't buffer hours of output
# into Python memory. Loop exits when:
#   - RESULTS.TXT appears (key found)
#   - proc exits on its own (error or completion)
#   - user interrupts the cell (Colab stop button → KeyboardInterrupt)
#   - Colab kernel killed (~2h on free tier — kills the loop too)
import datetime, signal, time

now = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = os.path.join(SESSIONS_DIR, f"{now}.log")
log_file = open(log_path, "w", buffering=1)
print(f"log: {log_path}")

cmd = [binary, "-gpu", "0",
       "-dp", str(DP_BITS),
       "-range", str(RANGE_BITS),
       "-start", START_HEX,
       "-pubkey", PUBKEY_HEX]
print("$", " ".join(cmd))

proc = subprocess.Popen(
    cmd, cwd=WORK_DIR, stdout=log_file, stderr=subprocess.STDOUT,
)

POLL_EVERY = 60        # 1 minute
HEARTBEAT  = 20 * 60   # print log tail every 20 min

def tail(path, n=20):
    if not os.path.exists(path):
        return ""
    with open(path) as f:
        lines = f.readlines()
    return "".join(lines[-n:])

start = time.monotonic()
last_heartbeat = 0.0
try:
    while proc.poll() is None:
        time.sleep(POLL_EVERY)
        elapsed = time.monotonic() - start

        if os.path.exists(LOCAL_RESULT):
            print(f"[+{elapsed/60:.1f}m] RESULTS.TXT detected, terminating")
            proc.send_signal(signal.SIGTERM)
            break

        if elapsed - last_heartbeat >= HEARTBEAT:
            print(f"[+{elapsed/60:.0f}m] heartbeat")
            print(tail(log_path, 6))
            last_heartbeat = elapsed
except KeyboardInterrupt:
    print("interrupted; SIGTERM")
    proc.send_signal(signal.SIGTERM)
    proc.wait(timeout=120)
finally:
    log_file.close()
    print(f"session ran for {(time.monotonic() - start)/3600:.2f}h, rc={proc.poll()}")

In [ ]:
# Verify any recovered key. Triple-check before declaring success.
import re
from kangaroo.verify import verify_solution

if not os.path.exists(LOCAL_RESULT):
    print("no RESULTS.TXT — solver did not find the key this session")
    print("re-run the notebook in a future session to continue searching")
else:
    text = open(LOCAL_RESULT).read()
    print(f"--- {LOCAL_RESULT} ---")
    print(text)
    print("---")

    candidate = None
    for pat in [
        r"Priv(?:ate)?\s*[Kk]ey\s*:\s*(?:0x)?([0-9a-fA-F]+)",
        r"Priv\s*:\s*(?:0x)?([0-9a-fA-F]+)",
        r"\b([0-9a-fA-F]{30,})\b",
    ]:
        m = re.search(pat, text)
        if m:
            candidate = int(m.group(1), 16)
            break
    if candidate is None:
        raise RuntimeError("RESULTS.TXT exists but no key parseable")

    print(f"candidate d = 0x{candidate:x}")
    verify_solution(candidate, PUBKEY_HEX, K1, K2)
    print(f"\nVERIFIED: d = 0x{candidate:x} solves puzzle #135")
    print(f"target address: {ADDRESS}")

    # Persist verified result to Drive separately from the in-progress file.
    drive_result = os.path.join(DRIVE_DIR, f"FOUND-{now}.txt")
    shutil.copy(LOCAL_RESULT, drive_result)
    print(f"\nresult archived: {drive_result}")
    print("\n!!  Do NOT broadcast a claim transaction over a public mempool.")
    print("    Front-running risk on puzzle prizes is real (~10% loss precedent).")
    print("    Use a private relay (e.g. Mara Slipstream) for any claim tx.")


## After the session

If a key was found: the verified result is in Drive at `kangaroo_135/FOUND-*.txt`.
**Do not broadcast a normal Bitcoin transaction with the key** — front-running
is a known risk. See the front-running note in the project CLAUDE.md.

If no key: re-run this notebook next session. Each session uses a different
seed for kangaroo starting positions, so consecutive runs explore independent
slices of the search space (per-session probability ≈ 3×10⁻⁸ at 2h on T4).

## What this notebook does NOT have yet

- A claim-transaction builder using a private relay. That's a separate piece
  of work and only matters if a key is actually found.